# P9 — 평가 반환 후 원본 cache 재현 감사

SAE 8개·TC 8개를 재학습하지 않습니다. 기존 평가의 feature·threshold·checkpoint·bin을 바꾸지 않고 원본 P5 cache에서 재현합니다.

입력 `v1_4_p9_source_audit_r1.zip`을 내 드라이브 `boolean_interp_v1_4/`에 업로드하고 **기존 평가와 같은 T4 GPU 런타임**에서 위부터 실행하세요.
기존 `P5_r2`, `P9_evaluation_r1/sae`, `P9_evaluation_r1/tc`를 읽고 새 결과를 `P9_source_audit_r1`에 저장합니다.

CPU probe refit와 bootstrap 때문에 수 시간이 걸릴 수 있습니다. 개별 refit·완료 run을 저장하며 같은 노트북으로 재개합니다. 환경 ID와 로그를 보존합니다.
완료 또는 실패 후 마지막 셀에서 ZIP·checksum을 내려받아 반환하세요. 실제 감사 반환을 확인하기 전에는 P9를 완료하지 않습니다.

In [ ]:
from google.colab import drive,files
from pathlib import Path
import hashlib,zipfile,subprocess,sys,time,os,json,shutil
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/boolean_interp_v1_4')
bundle=Path('/content/v1_4_p9_source_audit_r1.zip')
def checksum(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
    return h.hexdigest()
EXPECTED='c55f135be22567af3c0c8a2384b2e8d6e0480b4d4c51c16cba5411eae5c63b5a'
if not bundle.exists() or checksum(bundle)!=EXPECTED:shutil.copyfile(BASE/bundle.name,bundle)
assert checksum(bundle)==EXPECTED, '입력 checksum 불일치'
ROOT=Path('/content/MI_P9_source_audit_r1');ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle) as z:
    for n in z.namelist():assert (ROOT/n).resolve().is_relative_to(ROOT.resolve())
    z.extractall(ROOT)
SOURCE=BASE/'P5_r2';RETURNED=BASE/'P9_evaluation_r1';OUT=BASE/'P9_source_audit_r1';OUT.mkdir(exist_ok=True)
assert (SOURCE/'contract.json').exists()
assert all((RETURNED/t/'evaluation_complete.json').exists() for t in ('sae','tc')), '기존 평가 경로 확인 필요'


## 1. 수치 라이브러리·환경 준비
기존 평가 버전을 설치하고 설치 로그·환경 lock·GPU 정보를 보존합니다. 재시작 안내가 나오면 런타임을 재시작하고 첫 셀부터 실행하세요.

In [ ]:
os.environ['OMP_NUM_THREADS']='2';os.environ['OPENBLAS_NUM_THREADS']='2'
log=subprocess.run([sys.executable,'-m','pip','install','numpy==2.1.3','scipy==1.16.3','pytest==8.4.2','matplotlib==3.10.8','torch==2.11.0','--extra-index-url','https://download.pytorch.org/whl/cu128'],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
(OUT/f'install_{time.time_ns()}.txt').write_text(log.stdout);print(log.stdout);log.check_returncode()
subprocess.run([sys.executable,'-m','pytest','tests_v1_4/test_p9_source_audit.py','tests_v1_4/test_p9_sae_evaluation.py','tests_v1_4/test_p9_tc_evaluation.py','-q'],cwd=ROOT,check=True)


## 2. 원본 재현 감사

반환 파일 38,080개와 P5 원본 hash를 검증합니다. 두 도구 합계 560개 신규 고유 probe를 다시 fit하며, 재사용 포함 1,200 probe 작업의 선택·계수·threshold와 모든 test 예측·의미 CI를 대조합니다.
각 모델의 train/val/test 80,000 위치 fidelity와 train dead 비율, 전체 causal validation bins를 확인합니다.

GPU 패칭은 효과와 무관하게 고정된 suite별 첫 pair 4개와 general READ 첫 16개를 모델마다 재현합니다. 모든 인과·대체 CI, 층간 차이와 seed 0/1 비교표도 다시 계산합니다.
**전체 인과 본평가를 다시 실행하는 것이 아닙니다.** 불일치는 실패로 보존하고 선택값을 변경하지 않습니다. 단계별 로그는 OUT/logs에 저장됩니다.

In [ ]:
command=[sys.executable,'-u',str(ROOT/'scripts/audit_v1_4_p9_sources.py'),'--root',str(ROOT),'--source',str(SOURCE),'--returned',str(RETURNED),'--output',str(OUT)]
logs=OUT/'logs';logs.mkdir(exist_ok=True)
with (logs/f'audit_{time.time_ns()}.log').open('w',buffering=1) as log:
    proc=subprocess.Popen(command,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in proc.stdout:log.write(line);print(line,end='',flush=True)
        code=proc.wait()
        if code:raise subprocess.CalledProcessError(code,command)
    finally:
        if proc.poll() is None:
            proc.terminate()
            try:proc.wait(timeout=10)
            except subprocess.TimeoutExpired:proc.kill();proc.wait()
        proc.stdout.close()


## 3. 완료·중단 결과 다운로드
위 셀이 실패해도 실행할 수 있습니다. 아래 ZIP과 checksum JSON을 이 작업에 전달하세요. 기존 평가 파일은 수정하지 않습니다.

In [ ]:
archive=Path(f'/content/v1_4_p9_source_audit_return_{time.time_ns()}.zip')
subprocess.run([sys.executable,'-m','interp_v1_4.p9_sae_runner','export','--output',str(OUT),'--archive',str(archive)],cwd=ROOT,check=True)
files.download(str(archive));files.download(str(archive.with_suffix('.sha256.json')))
